# Cell 1 — Setup & load the saved checkpoint

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import set_seed
set_seed(42)

import os
PROJECT_ROOT = '/content/drive/MyDrive/Bengali Paraphrasing'
PROCESSED_DIR = f'{PROJECT_ROOT}/processed_data'
BEST_MODEL_DIR = f'{PROJECT_ROOT}/checkpoints/qwen2.5/best_model'
BASE_MODEL_CACHE = f'{PROJECT_ROOT}/base_model_cache/qwen2.5'

!pip install -q transformers accelerate bitsandbytes peft

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_CACHE, quantization_config=bnb_config, device_map={"": 0},
    attn_implementation="sdpa"
)
model = PeftModel.from_pretrained(base_model, BEST_MODEL_DIR)
model.eval()
print("Loaded Qwen2.5 best_model checkpoint successfully.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 28.6 MB/s eta 0:00:00


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Qwen2.5 best_model checkpoint successfully.


# Cell 2 — Load the FULL 5K validation set, preprocess identically to training

In [2]:
import pandas as pd
from datasets import Dataset

MAX_LENGTH = 512
INSTRUCTION = "নিচের বাক্যটি ভিন্নভাবে, একই অর্থ বজায় রেখে লিখুন।"

val_df_full = pd.read_parquet(os.path.join(PROCESSED_DIR, 'validation_subset_5k.parquet'))
print(f"Full validation set size: {len(val_df_full)}")  # should be 5000, confirming this is NOT the reduced 500 subset

val_ds_full = Dataset.from_pandas(val_df_full[['source', 'target']])

def preprocess(example):
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n\n{example['source']}"}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + example['target'] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)['input_ids']

    labels = full_ids.copy()
    prompt_len = min(len(prompt_ids), len(full_ids))
    labels[:prompt_len] = [-100] * prompt_len

    return {'input_ids': full_ids, 'attention_mask': [1] * len(full_ids), 'labels': labels}

val_tok_full = val_ds_full.map(preprocess, remove_columns=['source', 'target'])
print("Preprocessing complete. Sample:", val_tok_full[0])

Full validation set size: 5000


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Preprocessing complete. Sample: {'input_ids': [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 86548, 80178, 248, 58908, 72258, 35178, 105, 49128, 243, 52806, 107, 147338, 61356, 35178, 255, 80178, 101, 52806, 101, 148222, 49128, 105, 58908, 11, 35178, 237, 146775, 148868, 35178, 227, 72258, 52806, 98, 35178, 105, 148204, 49128, 107, 11125, 120, 35178, 108, 58908, 149526, 58908, 35178, 110, 80178, 244, 26927, 223, 86548, 146031, 271, 146711, 58908, 149526, 49128, 101, 58908, 148873, 35178, 243, 80178, 249, 26927, 223, 35178, 250, 147338, 80178, 110, 147645, 41312, 35178, 228, 148972, 58908, 11, 35178, 116, 58908, 148787, 26927, 223, 146227, 26927, 233, 35178, 228, 146227, 49128, 103, 35178, 228, 146227, 26927, 233, 148431, 86548, 41312, 35178, 243, 72258, 58908, 35178, 254, 80178, 243, 35178, 243, 72258, 147645, 58908, 35178, 117, 146026, 58908, 146031, 151645, 198, 151644, 77091, 198, 146775, 

# Cell 3 — Data collator (identical to training)

In [3]:
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class CausalLMPaddingCollator:
    pad_id: int
    def __call__(self, features: List[Dict]):
        def pad_seqs(key, pad_value):
            seqs = [f[key] for f in features]
            max_len = max(len(s) for s in seqs)
            return torch.tensor([s + [pad_value] * (max_len - len(s)) for s in seqs])
        return {
            'input_ids': pad_seqs('input_ids', self.pad_id),
            'attention_mask': pad_seqs('attention_mask', 0),
            'labels': pad_seqs('labels', -100),
        }

collator = CausalLMPaddingCollator(pad_id=tokenizer.pad_token_id)

# Cell 4 — Run genuine evaluation on the full 5K set

In [4]:
from transformers import TrainingArguments, Trainer

eval_args = TrainingArguments(
    output_dir='/content/qwen_full_eval_tmp',
    per_device_eval_batch_size=2,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=val_tok_full,
    data_collator=collator,
)

results = trainer.evaluate()
print("\n=== GENUINE Full 5K Validation Results (Qwen2.5-1.5B, best_model checkpoint) ===")
print(results)

Training Loss,Validation Loss,Step
No log,0.410903,0



=== GENUINE Full 5K Validation Results (Qwen2.5-1.5B, best_model checkpoint) ===
{'eval_loss': 0.41090282797813416}


# Cell 5 — Save this real result to disk, for the record

In [5]:
import json

full_eval_record = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "checkpoint": "best_model (early-stopped at step 2700, effective batch 16)",
    "evaluation_set": "validation_subset_5k.parquet (FULL 5,000 examples, matching TigerLLM/IndicBARTSS/BanglaT5)",
    "eval_loss": results.get('eval_loss'),
    "note": "This is a genuine post-hoc re-evaluation of the already-trained best_model checkpoint on the full validation set, for direct methodological comparability with TigerLLM. No training was performed; this is pure inference.",
}

record_path = f'{PROJECT_ROOT}/checkpoints/qwen2.5/full_5k_eval_result.json'
with open(record_path, 'w', encoding='utf-8') as f:
    json.dump(full_eval_record, f, ensure_ascii=False, indent=2)

print(json.dumps(full_eval_record, ensure_ascii=False, indent=2))
print("\nSaved to:", record_path)

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "checkpoint": "best_model (early-stopped at step 2700, effective batch 16)",
  "evaluation_set": "validation_subset_5k.parquet (FULL 5,000 examples, matching TigerLLM/IndicBARTSS/BanglaT5)",
  "eval_loss": 0.41090282797813416,
  "note": "This is a genuine post-hoc re-evaluation of the already-trained best_model checkpoint on the full validation set, for direct methodological comparability with TigerLLM. No training was performed; this is pure inference."
}

Saved to: /content/drive/MyDrive/Bengali Paraphrasing/checkpoints/qwen2.5/full_5k_eval_result.json
